# 🔬 Clase 1: Fundamentos de Inteligencia Artificial y Machine Learning
**Cátedra:** Minería de Datos y Aprendizaje Automático — Facultad de Informática, UNLP (2026)
**Alumno:** Agustín Barthe

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)

---
## 🎯 Objetivos de la Práctica
1. Comprender la distinción formal entre **IA Simbólica** (Deducción) e **IA Inductiva** (Machine Learning).
2. Implementar vectorialmente en **NumPy** las métricas fundamentales de distancia: Euclidiana ($), Manhattan ($) y Minkowski ($).
3. Construir desde cero un clasificador **K-Nearest Neighbors (KNN)** con la API de Scikit-Learn ( y ), soportando votación uniforme y ponderada por distancia.
4. Implementar estandarización hBcScore y analizar el impacto de la escala en la geometría euclidiana.


### 1. Inicialización y Dependencias
Importamos NumPy para el manejo de arreglos multidimensionales de alta eficiencia.

In [ ]:
import numpy as np
print(f"NumPy versión: {np.__version__}")

---
## 📐 Desafío 1: Métricas de Distancia Vectorizadas

### Fórmulas Matemáticas Desglosadas:
- **Distancia Euclidiana ($):**
20371d_2(\mathbf{p}, \mathbf{q}) = \sqrt{\sum_{i=1}^n (p_i - q_i)^2}20371

- **Distancia Manhattan ($):**
20371d_1(\mathbf{p}, \mathbf{q}) = \sum_{i=1}^n |p_i - q_i|20371

- **Distancia Minkowski ($):**
20371d_p(\mathbf{p}, \mathbf{q}) = \left( \sum_{i=1}^n |p_i - q_i|^p ight)^{1/p}20371


In [ ]:
def distancia_euclidiana(p, q):
    diff = np.asarray(p, dtype=float) - np.asarray(q, dtype=float)
    return float(np.sqrt(np.sum(diff ** 2)))

def distancia_manhattan(p, q):
    diff = np.asarray(p, dtype=float) - np.asarray(q, dtype=float)
    return float(np.sum(np.abs(diff)))

def distancia_minkowski(p, q, p_order=3):
    diff = np.asarray(p, dtype=float) - np.asarray(q, dtype=float)
    return float(np.sum(np.abs(diff) ** p_order) ** (1.0 / p_order))

# Tests de validación
p = [0, 0]
q = [3, 4]
assert np.isclose(distancia_euclidiana(p, q), 5.0), "Error en Euclidiana"
assert np.isclose(distancia_manhattan(p, q), 7.0), "Error en Manhattan"
assert np.isclose(distancia_minkowski(p, q, p_order=1), distancia_manhattan(p, q)), "Error equivalencia p=1"
assert np.isclose(distancia_minkowski(p, q, p_order=2), distancia_euclidiana(p, q)), "Error equivalencia p=2"
print("✓ Desafío 1 superado exitosamente.")

---
## ⚖️ Desafío 2: Estandarización y Escalamiento

### Fórmulas:
20371z = rac{x - \mu}{\sigma}, \quad \mu = rac{1}{N}\sum_{i=1}^N x_i, \quad \sigma = \sqrt{rac{1}{N}\sum_{i=1}^N (x_i - \mu)^2}20371


In [ ]:
def z_score_estandarizar(X):
    X_arr = np.asarray(X, dtype=float)
    mu = np.mean(X_arr, axis=0)
    sigma = np.std(X_arr, axis=0)
    sigma_safe = np.where(sigma == 0, 1.0, sigma)
    return (X_arr - mu) / sigma_safe, mu, sigma

# Test de validación
X_test = np.array([[10.0, 1000.0], [20.0, 2000.0], [30.0, 3000.0]])
Z, mu, sigma = z_score_estandarizar(X_test)
assert np.allclose(np.mean(Z, axis=0), [0.0, 0.0])
assert np.allclose(np.std(Z, axis=0), [1.0, 1.0])
print("✓ Desafío 2 superado exitosamente.")

---
## 🧠 Desafío 3: Clasificador KNN desde Cero ()

Implementación del modelo de memoria con soporte de ponderación por inverso de distancia:
20371w_i = rac{1}{d(\mathbf{x}, \mathbf{x}_i) + \epsilon}, \quad \hat{y} = rg\max_{c} \sum_{i: y_i = c} w_i20371


In [ ]:
class KNNClassifierScratch:
    def __init__(self, k=3, metric="euclidean", weights="uniform"):
        self.k = k
        self.metric = metric
        self.weights = weights
        self.X_train = None
        self.y_train = None

    def fit(self, X, y):
        self.X_train = np.asarray(X, dtype=float)
        self.y_train = np.asarray(y)
        return self

    def predict(self, X):
        X = np.asarray(X, dtype=float)
        return np.array([self._predict_single(x) for x in X])

    def _predict_single(self, x):
        if self.metric == "euclidean":
            distances = np.sqrt(np.sum((self.X_train - x) ** 2, axis=1))
        elif self.metric == "manhattan":
            distances = np.sum(np.abs(self.X_train - x), axis=1)
        else:
            raise ValueError(f"Métrica desconocida: {self.metric}")

        k_indices = np.argsort(distances)[:self.k]
        k_labels = self.y_train[k_indices]
        k_dists = distances[k_indices]

        if self.weights == "uniform":
            classes, counts = np.unique(k_labels, return_counts=True)
            return classes[np.argmax(counts)]
        elif self.weights == "distance":
            eps = 1e-6
            w = 1.0 / (k_dists + eps)
            scores = {}
            for label, weight in zip(k_labels, w):
                scores[label] = scores.get(label, 0.0) + weight
            return max(scores, key=scores.get)

# Test de clasificación toy
X_toy = np.array([[1.0, 1.0], [1.5, 1.2], [1.2, 1.8], [8.0, 8.0], [8.5, 8.2], [8.2, 8.8]])
y_toy = np.array([0, 0, 0, 1, 1, 1])
knn = KNNClassifierScratch(k=3)
knn.fit(X_toy, y_toy)
preds = knn.predict([[1.1, 1.2], [8.1, 8.3]])
assert np.array_equal(preds, [0, 1])
print("✓ Desafío 3 superado exitosamente.")

---
## 📚 Ejercicios Teóricos de Cátedra (Para Repaso y Examen)

1. **Searle vs. Turing:** ¿Por qué la manipulación formal de símbolos en un LLM no garantiza entendimiento semántico?
2. **Lazy Learning:** Analiza por qué el costo computacional de inferencia en KNN escala como $\mathcal{O}(N \cdot d)$.
3. **Maldición de la Dimensionalidad:** ¿Por qué la distancia euclidiana pierde poder de discriminación cuando  	o \infty127
